In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer, BertForQuestionAnswering, AdamW, get_scheduler
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import numpy as np
import collections
#from arabert.preprocess import ArabertPreprocessor

# Check if a GPU is available and set the device
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model_name="aubmindlab/bert-large-arabertv2"
#arabert_prep = ArabertPreprocessor(model_name=model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = BertForQuestionAnswering.from_pretrained(model_name, 
                                             #device_map='auto', 
                                             #load_in_8bit=True,
                                             #torch_dtype=torch.float16,
                                             #bnb_4bit_compute_dtype=torch.float16  # Set the compute data type to float16 for better performance
                                            ).to(device)
# Load the ARCD dataset from CSV
df = pd.read_csv(r'C:\Users\mosab\Desktop\Datasets\General\ARCD\arcd_all.csv', encoding='utf-16', sep='\t')

Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at aubmindlab/bert-large-arabertv2 and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [2]:
Greek_letters=list(range(913,938))+list(range(945,970))
Greek_letters.remove(930)
#Σ: σ/ς for capital sigma there are two alternatives small letters
Greek_letters=[chr(code) for code in Greek_letters]
German_letters=['ä','Ä','ü','Ü','ö','Ö','ß']
tokenizer.add_tokens(Greek_letters+German_letters)
# Resize the model's token embeddings to include the new tokens
model.resize_token_embeddings(len(tokenizer))

Embedding(64056, 1024)

In [3]:

# Split dataset into train, validation, and test sets
train_df, test_df = train_test_split(df, test_size=0.1)#, random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.1)#, random_state=42)

In [4]:
#train_df.question[0] = "من هو جمال أحمد حمزة خاشقجي"
#train_df.question[0] = "i am going to school"

# train_df.question[0]= arabert_prep.preprocess(train_df.question[0])
# train_df.context[0]=arabert_prep.preprocess(train_df.context[0])
# train_df.answer[0]=arabert_prep.preprocess(train_df.answer[0])
# train_df.title[0]=arabert_prep.preprocess(train_df.title[0])


In [5]:
# from arabert.preprocess import ArabertPreprocessor

# model_name="aubmindlab/bert-large-arabertv2"
# arabert_prep = ArabertPreprocessor(model_name=model_name)

# text = "ولن نبالغ إذا قلنا إن هاتف أو كمبيوتر المكتب في زمننا هذا ضروري"
# arabert_prep.preprocess(text)

In [6]:
import re

# Text normalization for comparison (lowercase, remove articles, punctuation, etc.)
def normalize_text(text):
    text = text.lower()
    #text = re.sub(r'\b(a|an|the)\b', ' ', text)  # Remove articles
    text = re.sub(r'[\u064B-\u065F]', '', text)  # Remove diacritics
    text = re.sub(r'\W+', ' ', text)  # Remove punctuation and double whitespaces
    #text = re.sub(r'[^a-zء-ي0-9]+', ' ', text)  # Remove double whitespaces
    #text = ' '.join(text.split())  # Remove extra spaces
    return text

# Exact Match (EM) calculation
def exact_match_score(predicted_answer, actual_answer):
    return int(normalize_text(predicted_answer) == normalize_text(actual_answer))

# F1 Score calculation for a single prediction
def f1_single(predicted_answer, actual_answer):
    # Tokenize predicted and actual answers
    pred_tokens = normalize_text(predicted_answer).split()
    actual_tokens = normalize_text(actual_answer).split()
    
    if (len(pred_tokens) == 0) and (len(actual_tokens) == 0):
        return 1
    elif (len(pred_tokens) == 0):
        return 0
    elif (len(actual_tokens) == 0):
        return 0
     #common_tokens = pred_tokens.intersection(actual_tokens)
    common_tokens = set(pred_tokens) & set(actual_tokens)
    #common_tokens = remove_stopwords_from_set(common_tokens, stopwords_list)
    if len(common_tokens) == 0:
        return 0.0
    
    precision = len(common_tokens) / len(pred_tokens)
    recall = len(common_tokens) / len(actual_tokens)
    f1 = 2 * (precision * recall) / (precision + recall)
    return f1

# Function to calculate average EM and F1 scores across the dataset
def evaluate_qa(predicted_answers, actual_answers):
    total_em = 0
    total_f1 = 0
    n = len(predicted_answers)
    
    for predicted, actual in zip(predicted_answers, actual_answers):
        total_em += exact_match_score(predicted, actual)
        total_f1 += f1_single(predicted, actual)
    
    # Calculate the average EM and F1 scores
    em_score = total_em / n
    f1_score_avg = total_f1 / n
    return em_score, f1_score_avg

In [7]:
# Assuming 'context_tokens' is a list of tokens from the context
# and 'answer_tokens' is a list of tokens from the answer
def find_answer_span(context_tokens, answer_tokens):
    #we added '+1' in 'range(len(context_tokens) - len(answer_tokens) + 1)', 
    #becuase in 'context_tokens[i:i + len(answer_tokens)]' the index at 'i' is included 
    #and the index at 'i + len(answer_tokens)' is excluded 
    for i in range(len(context_tokens) - len(answer_tokens) + 1):
        # Check if the slice of context tokens matches the answer tokens
        if context_tokens[i:i + len(answer_tokens)] == answer_tokens:
            return i, i + len(answer_tokens) -1
    return None, None  # Return None if the answer is not found

In [8]:
# Define max length and stride for sliding window
max_length = 512  # Define input size for the model
doc_stride = 128  # Sliding window size

# Define custom dataset class
class ARCDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        example = self.data.iloc[idx]
        question = example['question']
        context = example['context']
        title = example['title']
        answer = example['answer']
#         print("answer")
#         print(answer)
#         print("context")
#         print(context)
        
        
        # Tokenize question and context with truncation
        encoding = tokenizer(
            example['question'],
            example['context'],
            max_length=max_length,
            truncation=True,
            padding="max_length",
            stride=doc_stride,
            return_overflowing_tokens=False,  # Handles long inputs
            return_offsets_mapping=False,    # Skipping offset mapping to avoid char-to-token problems
            return_tensors="pt"
        )
        
        
        # Tokenize answer separately to align with context
        answer_tokens = tokenizer.encode(answer, add_special_tokens=False)
        question_tokens = tokenizer.encode(question, add_special_tokens=False)
        context_tokens = tokenizer.encode(context, add_special_tokens=False)
        
        length_first_SEP = len(question_tokens)+2 #for '[CLS]' and '[SEP]'
#         print("length_first_SEP")
#         print(length_first_SEP)
        
        # Find answer token positions in the context tokens
        answer_start_idx, answer_end_idx = find_answer_span(context_tokens, answer_tokens)
        answer_start_idx+=length_first_SEP
        answer_end_idx+=length_first_SEP
        #answer_start_idx = context_tokens.index(answer_tokens[0])+length_first_SEP
        #answer_end_idx = answer_start_idx + len(answer_tokens)
        
#         print("answer_start_idx")
#         print(answer_start_idx)
#         print("answer_end_idx")
#         print(answer_end_idx)
        

        # Handle cases where the answer is not in the tokenized input
        if answer_start_idx is None:
            answer_start_idx = self.max_length - 1
        if answer_end_idx is None:
            answer_end_idx = self.max_length - 1
        
#         print("tokenizer.decode(answer_tokens, skip_special_tokens=True)")
#         print(tokenizer.decode(answer_tokens, skip_special_tokens=True))
#         print("answer_tokens")
#         print(answer_tokens)
#         print("answer_start_idx")
#         print(answer_start_idx)
#         print("answer_end_idx")
#         print(answer_end_idx)
#         #print("encoding['input_ids'].squeeze()")
#         input_ids=encoding['input_ids'].squeeze()
#         #print(input_ids)
#         tokens = tokenizer.convert_ids_to_tokens(input_ids)
#         # Slice the token IDs between start_index and end_index
#         token_ids_slice = input_ids[answer_start_idx:answer_end_idx]
#         # Convert the sliced token IDs to token strings
#         tokens_slice = tokenizer.convert_ids_to_tokens(token_ids_slice)
#         print("tokens_slice")
#         print(tokens_slice)
#         restored_answer=tokenizer.decode(token_ids_slice, skip_special_tokens=True)
#         print("restored_answer")
#         print(restored_answer)
#         print("tokens")
#         print(tokens)
        
        
        
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'token_type_ids': encoding['token_type_ids'].squeeze(),
            'start_positions': torch.tensor(answer_start_idx),
            'end_positions': torch.tensor(answer_end_idx),
#             'question':question,
#             'answer':answer
        }

# Create DataLoaders for train, val, and test sets
train_dataset = ARCDataset(train_df, tokenizer)
val_dataset = ARCDataset(val_df, tokenizer)
test_dataset = ARCDataset(test_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)


# Optimizer and learning rate scheduler
optimizer = AdamW(model.parameters(), lr=3e-5)
num_epochs = 3
num_training_steps = num_epochs * len(train_loader)
lr_scheduler = get_scheduler(
    name="linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)

# Training function
def train(model, dataloader):
    model.train()
    total_loss = 0
    last_loss=0
    for batch in tqdm(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch['token_type_ids'].to(device)
        start_positions = batch['start_positions'].to(device)
        end_positions = batch['end_positions'].to(device)

#         print("input_ids")
#         print(input_ids)
#         print("attention_mask")
#         print(attention_mask)
#         print("token_type_ids")
#         print(token_type_ids)
#         print("start_positions")
#         print(start_positions)
#         print("end_positions")
#         print(end_positions)
        
#         input_ids = batch['input_ids'].to(device)
#         attention_mask = batch['attention_mask'].to(device)
#         token_type_ids = batch['token_type_ids'].to(device)
#         start_positions = batch['start_positions'].to(device)
#         end_positions = batch['end_positions'].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            start_positions=start_positions,
            end_positions=end_positions
        )
        #print("outputs")
        #print(outputs)
        
        loss = outputs.loss
        loss.backward()
        print("loss")
        print(loss)
        
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        
        last_loss=loss.item()
        total_loss += loss.item()
        
    print(f"Last loss: {last_loss:.4f}") 
    return total_loss / len(dataloader)

# Evaluation function
def evaluate(model, dataloader):
    model.eval()
    
    predicted_answers=[]
    actual_answers=[]
    
    for batch in tqdm(dataloader):
        #print(batch.keys())
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch['token_type_ids'].to(device)
#         print("input_ids")
#         print(input_ids)
#         print("attention_mask")
#         print(attention_mask)
#         print("token_type_ids")
#         print(token_type_ids)
        
        with torch.no_grad():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )
        
        start_logits = outputs.start_logits
        end_logits = outputs.end_logits
#         print("start_logits")
#         print(start_logits)
#         print("end_logits")
#         print(end_logits)
        
#         start_preds = torch.argmax(start_logits, dim=1)
#         end_preds = torch.argmax(end_logits, dim=1)
        
    
    
        
        start_preds=[]
        end_preds=[]
        
        # Define a threshold below which we consider there is no answer
        no_answer_threshold = 2.0  # You can experiment with this value

        
        # Loop through each batch
        for i in range(start_logits.size(0)):  # Iterate over the batch size
            best_start_index = None
            best_end_index = None
            best_score = float('-inf')  # Initialize with a very low score

            # Get the logits for the i-th example
            start_logit = start_logits[i]  # Shape: [seq_length]
            end_logit = end_logits[i]      # Shape: [seq_length]

            # Get the indices of the top 5 highest start and end logits
            top_5_start_indices = torch.topk(start_logit, 5).indices.tolist()
            top_5_end_indices = torch.topk(end_logit, 5).indices.tolist()

            # Loop through the top 10 start and top 10 end indices ensuring start < end
            for start_index in top_5_start_indices:
                for end_index in top_5_end_indices:
                    if start_index < end_index:  # Ensure start < end
                        score = start_logit[start_index] + end_logit[end_index]
                        if score > best_score:
                            best_start_index = start_index
                            best_end_index = end_index
                            best_score = score

            # Check if the best score is below the no-answer threshold
            if best_score < no_answer_threshold:
                print(f"No answer found for example {i} with score {best_score}, as the score is below the threshold.")
                best_start_index = None
                best_end_index = None
            else:
                print(f"Best start index for example {i}: {best_start_index}")
                print(f"Best end index for example {i}: {best_end_index}")

            # Optionally, return or print no answer if no valid start-end pair is found
            if best_start_index is None or best_end_index is None:
                print(f"No valid start-end pair found for example {i}.")
            else:
                print(f"Answer for example {i} starts at {best_start_index} and ends at {best_end_index}.")
        
            start_preds.append(best_start_index)
            end_preds.append(best_end_index)
    
    
    
    
    
        
        

#         print("start_preds")
#         print(start_preds)
#         print("end_preds")
#         print(end_preds)
#         # Convert predictions to strings and compare
#         print("len(batch['input_ids']")
#         print(len(batch['input_ids']))
              
        for i in range(len(batch['input_ids'])):
            try:
                pred_tokens = batch['input_ids'][i][start_preds[i]:end_preds[i] + 1]
                pred_answer = tokenizer.decode(pred_tokens, skip_special_tokens=True)
            except:
                pred_answer=''
            
            try:
                true_answer = batch['input_ids'][i][batch['start_positions'][i]:batch['end_positions'][i]+1]
                true_answer = tokenizer.decode(true_answer, skip_special_tokens=True)
            except:
                true_answer=''
            
            print("pred_answer")
            print(pred_answer)
#             print("batch['start_positions'][i]")
#             print(batch['start_positions'][i])
#             print("batch['end_positions'][i]")
#             print(batch['end_positions'][i])
            print("decode true_answer")
            print(true_answer)
#             print("answer")
#             print(batch['answer'][i])
            
            predicted_answers.append(pred_answer)
            actual_answers.append(true_answer)
    
#     for actual_answer, predicted_answer in list(zip(actual_answers, predicted_answers)):
#         print("actual_answer")
#         print(actual_answer)
#         print("predicted_answer")
#         print(predicted_answer)
#         print(">>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>")
        
    exact_match_score, avg_f1_score = evaluate_qa(predicted_answers, actual_answers)

    return exact_match_score, avg_f1_score




C:\Users\mosab\anaconda3\envs\tf\lib\site-packages\transformers\optimization.py:588: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [9]:
# Training loop
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")
    train_loss = train(model, train_loader)
    print(f"Train loss: {train_loss:.4f}")
    val_exact_match, val_f1 = evaluate(model, val_loader)
    print(f"Validation Exact Match: {val_exact_match:.4f}, F1: {val_f1:.4f}")

# Test the model on test set
test_exact_match, test_f1 = evaluate(model, test_loader)
print(f"Test Exact Match: {test_exact_match:.4f}, F1: {test_f1:.4f}")

Epoch 1/3


  0%|          | 0/142 [00:00<?, ?it/s]

C:\Users\mosab\anaconda3\envs\tf\lib\site-packages\transformers\models\bert\modeling_bert.py:435: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


loss
tensor(6.1280, device='cuda:0', grad_fn=<DivBackward0>)
loss
tensor(6.3642, device='cuda:0', grad_fn=<DivBackward0>)
loss
tensor(5.9819, device='cuda:0', grad_fn=<DivBackward0>)
loss
tensor(5.7216, device='cuda:0', grad_fn=<DivBackward0>)
loss
tensor(5.5140, device='cuda:0', grad_fn=<DivBackward0>)
loss
tensor(5.3882, device='cuda:0', grad_fn=<DivBackward0>)
loss
tensor(4.9519, device='cuda:0', grad_fn=<DivBackward0>)
loss
tensor(4.9943, device='cuda:0', grad_fn=<DivBackward0>)
loss
tensor(4.7001, device='cuda:0', grad_fn=<DivBackward0>)
loss
tensor(4.3048, device='cuda:0', grad_fn=<DivBackward0>)
loss
tensor(4.4181, device='cuda:0', grad_fn=<DivBackward0>)


Token indices sequence length is longer than the specified maximum sequence length for this model (569 > 512). Running this sequence through the model will result in indexing errors


...

  0%|          | 0/18 [00:00<?, ?it/s]

Best start index for example 0: 70
Best end index for example 0: 124
Answer for example 0 starts at 70 and ends at 124.
No answer found for example 1 with score 1.9478952884674072, as the score is below the threshold.
No valid start-end pair found for example 1.
No answer found for example 2 with score -0.41174668073654175, as the score is below the threshold.
No valid start-end pair found for example 2.
No answer found for example 3 with score -5.611190319061279, as the score is below the threshold.
No valid start-end pair found for example 3.
Best start index for example 4: 18
Best end index for example 4: 21
Answer for example 4 starts at 18 and ends at 21.
Best start index for example 5: 17
Best end index for example 5: 23
Answer for example 5 starts at 17 and ends at 23.
Best start index for example 6: 9
Best end index for example 6: 13
Answer for example 6 starts at 9 and ends at 13.
Best start index for example 7: 31
Best end index for example 7: 34
Answer for example 7 starts a

Best start index for example 0: 139
Best end index for example 0: 154
Answer for example 0 starts at 139 and ends at 154.
Best start index for example 1: 12
Best end index for example 1: 16
Answer for example 1 starts at 12 and ends at 16.
Best start index for example 2: 21
Best end index for example 2: 25
Answer for example 2 starts at 21 and ends at 25.
Best start index for example 3: 58
Best end index for example 3: 69
Answer for example 3 starts at 58 and ends at 69.
Best start index for example 4: 34
Best end index for example 4: 37
Answer for example 4 starts at 34 and ends at 37.
Best start index for example 5: 126
Best end index for example 5: 132
Answer for example 5 starts at 126 and ends at 132.
Best start index for example 6: 21
Best end index for example 6: 35
Answer for example 6 starts at 21 and ends at 35.
No answer found for example 7 with score 1.9616589546203613, as the score is below the threshold.
No valid start-end pair found for example 7.
pred_answer
بالجدري ولم

No answer found for example 0 with score 1.8636510372161865, as the score is below the threshold.
No valid start-end pair found for example 0.
Best start index for example 1: 286
Best end index for example 1: 290
Answer for example 1 starts at 286 and ends at 290.
Best start index for example 2: 26
Best end index for example 2: 31
Answer for example 2 starts at 26 and ends at 31.
Best start index for example 3: 217
Best end index for example 3: 227
Answer for example 3 starts at 217 and ends at 227.
Best start index for example 4: 19
Best end index for example 4: 28
Answer for example 4 starts at 19 and ends at 28.
Best start index for example 5: 132
Best end index for example 5: 153
Answer for example 5 starts at 132 and ends at 153.
Best start index for example 6: 83
Best end index for example 6: 169
Answer for example 6 starts at 83 and ends at 169.
Best start index for example 7: 193
Best end index for example 7: 195
Answer for example 7 starts at 193 and ends at 195.
pred_answer



Best start index for example 0: 49
Best end index for example 0: 54
Answer for example 0 starts at 49 and ends at 54.
Best start index for example 1: 35
Best end index for example 1: 39
Answer for example 1 starts at 35 and ends at 39.
Best start index for example 2: 79
Best end index for example 2: 82
Answer for example 2 starts at 79 and ends at 82.
Best start index for example 3: 73
Best end index for example 3: 80
Answer for example 3 starts at 73 and ends at 80.
Best start index for example 4: 53
Best end index for example 4: 57
Answer for example 4 starts at 53 and ends at 57.
Best start index for example 5: 180
Best end index for example 5: 188
Answer for example 5 starts at 180 and ends at 188.
Best start index for example 6: 108
Best end index for example 6: 124
Answer for example 6 starts at 108 and ends at 124.
No answer found for example 7 with score -1.41776704788208, as the score is below the threshold.
No valid start-end pair found for example 7.
pred_answer
الخليفة عمر 

In [10]:
# Evaluation function
              
train_dataset = ARCDataset(train_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

# Test the model on train set
test_exact_match, test_f1 = evaluate(model, train_loader)
print(f"Test Exact Match: {test_exact_match:.4f}, F1: {test_f1:.4f}")

  0%|          | 0/142 [00:00<?, ?it/s]

Best start index for example 0: 140
Best end index for example 0: 143
Answer for example 0 starts at 140 and ends at 143.
Best start index for example 1: 66
Best end index for example 1: 69
Answer for example 1 starts at 66 and ends at 69.
Best start index for example 2: 115
Best end index for example 2: 120
Answer for example 2 starts at 115 and ends at 120.
Best start index for example 3: 39
Best end index for example 3: 42
Answer for example 3 starts at 39 and ends at 42.
Best start index for example 4: 140
Best end index for example 4: 141
Answer for example 4 starts at 140 and ends at 141.
No answer found for example 5 with score -3.136716365814209, as the score is below the threshold.
No valid start-end pair found for example 5.
Best start index for example 6: 113
Best end index for example 6: 118
Answer for example 6 starts at 113 and ends at 118.
Best start index for example 7: 72
Best end index for example 7: 78
Answer for example 7 starts at 72 and ends at 78.
pred_answer
59 

Best start index for example 0: 19
Best end index for example 0: 22
Answer for example 0 starts at 19 and ends at 22.
Best start index for example 1: 73
Best end index for example 1: 85
Answer for example 1 starts at 73 and ends at 85.
Best start index for example 2: 55
Best end index for example 2: 60
Answer for example 2 starts at 55 and ends at 60.
Best start index for example 3: 166
Best end index for example 3: 168
Answer for example 3 starts at 166 and ends at 168.
Best start index for example 4: 116
Best end index for example 4: 129
Answer for example 4 starts at 116 and ends at 129.
Best start index for example 5: 20
Best end index for example 5: 23
Answer for example 5 starts at 20 and ends at 23.
Best start index for example 6: 101
Best end index for example 6: 104
Answer for example 6 starts at 101 and ends at 104.
Best start index for example 7: 106
Best end index for example 7: 115
Answer for example 7 starts at 106 and ends at 115.
pred_answer
هو مذهب إسلامي.
decode true_

Best start index for example 0: 36
Best end index for example 0: 37
Answer for example 0 starts at 36 and ends at 37.
Best start index for example 1: 25
Best end index for example 1: 28
Answer for example 1 starts at 25 and ends at 28.
Best start index for example 2: 164
Best end index for example 2: 174
Answer for example 2 starts at 164 and ends at 174.
Best start index for example 3: 73
Best end index for example 3: 138
Answer for example 3 starts at 73 and ends at 138.
Best start index for example 4: 110
Best end index for example 4: 113
Answer for example 4 starts at 110 and ends at 113.
Best start index for example 5: 47
Best end index for example 5: 72
Answer for example 5 starts at 47 and ends at 72.
Best start index for example 6: 231
Best end index for example 6: 237
Answer for example 6 starts at 231 and ends at 237.
Best start index for example 7: 19
Best end index for example 7: 65
Answer for example 7 starts at 19 and ends at 65.
pred_answer
عام 1981
decode true_answer
عا

Best start index for example 0: 177
Best end index for example 0: 180
Answer for example 0 starts at 177 and ends at 180.
Best start index for example 1: 150
Best end index for example 1: 151
Answer for example 1 starts at 150 and ends at 151.
Best start index for example 2: 20
Best end index for example 2: 46
Answer for example 2 starts at 20 and ends at 46.
Best start index for example 3: 52
Best end index for example 3: 67
Answer for example 3 starts at 52 and ends at 67.
Best start index for example 4: 74
Best end index for example 4: 94
Answer for example 4 starts at 74 and ends at 94.
Best start index for example 5: 36
Best end index for example 5: 51
Answer for example 5 starts at 36 and ends at 51.
No answer found for example 6 with score -0.3668065667152405, as the score is below the threshold.
No valid start-end pair found for example 6.
Best start index for example 7: 27
Best end index for example 7: 30
Answer for example 7 starts at 27 and ends at 30.
pred_answer
100 مليون 

...

Best start index for example 0: 114
Best end index for example 0: 118
Answer for example 0 starts at 114 and ends at 118.
Best start index for example 1: 99
Best end index for example 1: 121
Answer for example 1 starts at 99 and ends at 121.
Best start index for example 2: 66
Best end index for example 2: 70
Answer for example 2 starts at 66 and ends at 70.
No answer found for example 3 with score 1.6394649744033813, as the score is below the threshold.
No valid start-end pair found for example 3.
Best start index for example 4: 83
Best end index for example 4: 99
Answer for example 4 starts at 83 and ends at 99.
Best start index for example 5: 202
Best end index for example 5: 214
Answer for example 5 starts at 202 and ends at 214.
Best start index for example 6: 60
Best end index for example 6: 64
Answer for example 6 starts at 60 and ends at 64.
Best start index for example 7: 47
Best end index for example 7: 50
Answer for example 7 starts at 47 and ends at 50.
pred_answer
بالإسباني

Best start index for example 0: 70
Best end index for example 0: 187
Answer for example 0 starts at 70 and ends at 187.
Best start index for example 1: 156
Best end index for example 1: 157
Answer for example 1 starts at 156 and ends at 157.
Best start index for example 2: 30
Best end index for example 2: 48
Answer for example 2 starts at 30 and ends at 48.
Best start index for example 3: 191
Best end index for example 3: 220
Answer for example 3 starts at 191 and ends at 220.
Best start index for example 4: 142
Best end index for example 4: 155
Answer for example 4 starts at 142 and ends at 155.
Best start index for example 5: 179
Best end index for example 5: 187
Answer for example 5 starts at 179 and ends at 187.
Best start index for example 6: 21
Best end index for example 6: 37
Answer for example 6 starts at 21 and ends at 37.
Best start index for example 7: 47
Best end index for example 7: 64
Answer for example 7 starts at 47 and ends at 64.
pred_answer
وهي أول عملة رقمية لامركزي

Best start index for example 0: 106
Best end index for example 0: 111
Answer for example 0 starts at 106 and ends at 111.
No answer found for example 1 with score 1.814791202545166, as the score is below the threshold.
No valid start-end pair found for example 1.
Best start index for example 2: 216
Best end index for example 2: 221
Answer for example 2 starts at 216 and ends at 221.
Best start index for example 3: 75
Best end index for example 3: 77
Answer for example 3 starts at 75 and ends at 77.
Best start index for example 4: 30
Best end index for example 4: 36
Answer for example 4 starts at 30 and ends at 36.
Best start index for example 5: 28
Best end index for example 5: 30
Answer for example 5 starts at 28 and ends at 30.
Best start index for example 6: 12
Best end index for example 6: 65
Answer for example 6 starts at 12 and ends at 65.
Best start index for example 7: 83
Best end index for example 7: 92
Answer for example 7 starts at 83 and ends at 92.
pred_answer
11 مدينة روس

Best start index for example 0: 30
Best end index for example 0: 35
Answer for example 0 starts at 30 and ends at 35.
Best start index for example 1: 164
Best end index for example 1: 169
Answer for example 1 starts at 164 and ends at 169.
Best start index for example 2: 118
Best end index for example 2: 127
Answer for example 2 starts at 118 and ends at 127.
Best start index for example 3: 28
Best end index for example 3: 92
Answer for example 3 starts at 28 and ends at 92.
No answer found for example 4 with score 1.7831348180770874, as the score is below the threshold.
No valid start-end pair found for example 4.
Best start index for example 5: 89
Best end index for example 5: 104
Answer for example 5 starts at 89 and ends at 104.
Best start index for example 6: 120
Best end index for example 6: 139
Answer for example 6 starts at 120 and ends at 139.
Best start index for example 7: 26
Best end index for example 7: 29
Answer for example 7 starts at 26 and ends at 29.
pred_answer
54, 0

Best start index for example 0: 116
Best end index for example 0: 123
Answer for example 0 starts at 116 and ends at 123.
pred_answer
« صلى الله عليه وسلم »
decode true_answer
« صلى الله عليه وسلم »
Test Exact Match: 0.4367, F1: 0.7256


In [11]:
# Define directory to save the model and tokenizer
save_directory = r'D:\Jupyter Notebook\Research\models\bert-large-arabertv2 training with Greek and German letters'

# Save model and tokenizer
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

print(f"Model and tokenizer saved to {save_directory}")

Model and tokenizer saved to C:\Users\mosab\Desktop\Jupyter Notebook\Research\models\bert-large-arabertv2 training with Greek and German letters
